# Historical Data Analysis Template

Use this notebook to load, explore, and analyse historical crypto OHLCV data stored in our DuckDB database.

---

## Before You Start

Make sure you have already run **`04_multi_coin_1m_loader.ipynb`** to populate the database.  
The database file lives at: `../data/ohlcv_1m.duckdb`

**What's in the database?**
- 66 coins from the Roostoo exchange
- 1-minute candles for the past 30 days
- All data is in one table called `ohlcv`, in long format (one row = one candle)

**Table columns:**
| Column | Type | Description |
|--------|------|-------------|
| `ts` | timestamp | Candle open time (UTC) |
| `symbol` | string | Coin, e.g. `BTC-USD` |
| `interval` | string | Timeframe, e.g. `1m` |
| `open` | float | Open price |
| `high` | float | High price |
| `low` | float | Low price |
| `close` | float | Close price |
| `volume` | float | Volume traded |
| `source` | string | Data source (always `binance`) |

---
## 0. Imports

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Connect to the database (read-only so you can't accidentally overwrite anything)
con = duckdb.connect('../data/ohlcv_1m.duckdb', read_only=True)

print('Connected to database.')

---
## 1. See What's Available

Run this first to check which coins are in the database and how many rows each has.

In [ ]:
# Summary of all coins in the database
summary = con.execute("""
    SELECT
        symbol,
        COUNT(*)            AS rows,
        MIN(ts)             AS earliest,
        MAX(ts)             AS latest
    FROM ohlcv
    GROUP BY symbol
    ORDER BY symbol
""").fetchdf()

print(f'{len(summary)} coins available, {summary["rows"].sum():,} total rows')
summary

---
## 2. Load a Single Coin

Change `SYMBOL` below to whichever coin you want to analyse.  
Format is always `TICKER-USD` (e.g. `ETH-USD`, `SOL-USD`, `DOGE-USD`).

In [ ]:
# ── CHANGE THIS ──────────────────────────────────────────────────────────────
SYMBOL = 'BTC-USD'
# ─────────────────────────────────────────────────────────────────────────────

df = con.execute("""
    SELECT ts, open, high, low, close, volume
    FROM ohlcv
    WHERE symbol = ?
    ORDER BY ts
""", [SYMBOL]).fetchdf()

# Set ts as the index for easy time-series operations
df = df.set_index('ts')

print(f'Loaded {len(df):,} rows for {SYMBOL}')
print(f'Range: {df.index.min()} → {df.index.max()}')
df.head()

---
## 3. Filter by Date Range (Optional)

If you only want data from a specific period, use the cell below.  
Skip this if you want all 30 days.

In [ ]:
# ── CHANGE THESE ─────────────────────────────────────────────────────────────
START = '2026-03-01'
END   = '2026-03-18'
# ─────────────────────────────────────────────────────────────────────────────

df_filtered = df.loc[START:END]

print(f'Filtered to {len(df_filtered):,} rows between {START} and {END}')
df_filtered.head()

---
## 4. Resample to a Different Timeframe (Optional)

The raw data is 1-minute candles. You can resample to any timeframe.  
Common options: `'5min'`, `'15min'`, `'1H'`, `'4H'`, `'1D'`

In [ ]:
# ── CHANGE THIS ──────────────────────────────────────────────────────────────
TIMEFRAME = '1H'   # resample 1m candles into 1-hour candles
# ─────────────────────────────────────────────────────────────────────────────

df_resampled = df['close'].resample(TIMEFRAME).ohlc()   # OHLC from close prices
df_resampled['volume'] = df['volume'].resample(TIMEFRAME).sum()

print(f'Resampled to {TIMEFRAME}: {len(df_resampled):,} candles')
df_resampled.head()

---
## 5. Basic Price Chart

In [ ]:
# Plot the close price and volume for your chosen coin
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7),
                                gridspec_kw={'height_ratios': [3, 1]},
                                sharex=True)

ax1.plot(df.index, df['close'], linewidth=0.8, color='#2196F3')
ax1.set_title(f'{SYMBOL} — 1m Close Price', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.2f}'))

ax2.bar(df.index, df['volume'], color='#90CAF9', alpha=0.7, width=0.0007)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Time (UTC)')
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

plt.tight_layout()
plt.show()

---
## 6. Common Technical Indicators

These are the building blocks for most strategies. Add whichever you need.

In [ ]:
# Work on a copy so you don't mutate the original
dfc = df.copy()

# ── Moving Averages ───────────────────────────────────────────────────────────
dfc['ma10']  = dfc['close'].rolling(10).mean()
dfc['ma20']  = dfc['close'].rolling(20).mean()
dfc['ma50']  = dfc['close'].rolling(50).mean()
dfc['ema12'] = dfc['close'].ewm(span=12, adjust=False).mean()
dfc['ema26'] = dfc['close'].ewm(span=26, adjust=False).mean()

# ── MACD ─────────────────────────────────────────────────────────────────────
dfc['macd']        = dfc['ema12'] - dfc['ema26']
dfc['macd_signal'] = dfc['macd'].ewm(span=9, adjust=False).mean()
dfc['macd_hist']   = dfc['macd'] - dfc['macd_signal']

# ── RSI (14) ─────────────────────────────────────────────────────────────────
delta = dfc['close'].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
rs    = gain / loss.replace(0, float('nan'))
dfc['rsi14'] = 100 - (100 / (1 + rs))

# ── Bollinger Bands (20, 2σ) ──────────────────────────────────────────────────
dfc['bb_mid']   = dfc['close'].rolling(20).mean()
dfc['bb_std']   = dfc['close'].rolling(20).std()
dfc['bb_upper'] = dfc['bb_mid'] + 2 * dfc['bb_std']
dfc['bb_lower'] = dfc['bb_mid'] - 2 * dfc['bb_std']
dfc['bb_width'] = (dfc['bb_upper'] - dfc['bb_lower']) / dfc['bb_mid']

# ── ATR (14) ─────────────────────────────────────────────────────────────────
tr = pd.concat([
    dfc['high'] - dfc['low'],
    (dfc['high'] - dfc['close'].shift()).abs(),
    (dfc['low']  - dfc['close'].shift()).abs(),
], axis=1).max(axis=1)
dfc['atr14'] = tr.rolling(14).mean()

# ── Returns ───────────────────────────────────────────────────────────────────
dfc['returns']    = dfc['close'].pct_change()
dfc['log_return'] = np.log(dfc['close'] / dfc['close'].shift())

print('Indicators calculated. Columns available:')
print(list(dfc.columns))
dfc.tail(3)

---
## 7. Load Multiple Coins

If your analysis involves more than one coin (e.g. correlation, portfolio), load them here.

In [ ]:
# ── CHANGE THIS ──────────────────────────────────────────────────────────────
SYMBOLS = ['BTC-USD', 'ETH-USD', 'SOL-USD']
# ─────────────────────────────────────────────────────────────────────────────

# Load all coins, then pivot to wide format (one column per coin)
raw = con.execute("""
    SELECT ts, symbol, close
    FROM ohlcv
    WHERE symbol IN ({placeholders})
    ORDER BY ts
""".format(placeholders=','.join(['?'] * len(SYMBOLS))), SYMBOLS).fetchdf()

# Pivot: rows = timestamps, columns = coin symbols
prices = raw.pivot(index='ts', columns='symbol', values='close')

print(f'Loaded {len(prices):,} timestamps x {len(prices.columns)} coins')
prices.tail(3)

---
## 8. Correlation Matrix

How correlated are your selected coins? Useful for portfolio construction.

In [ ]:
# Correlation of 1m returns between coins
returns = prices.pct_change().dropna()
corr    = returns.corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ticks = range(len(corr.columns))
ax.set_xticks(ticks); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(ticks); ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)

ax.set_title('Return Correlation Matrix (1m)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Export to CSV

If you want to work with the data outside of this notebook (e.g. Excel, another script), export it here.

In [ ]:
# Export your working DataFrame to CSV
# Change the filename to something descriptive
OUTPUT_PATH = f'../data/{SYMBOL}_analysis.csv'

dfc.to_csv(OUTPUT_PATH)
print(f'Saved {len(dfc):,} rows to {OUTPUT_PATH}')

---
## 10. Close Connection

Always run this when you're done.

In [ ]:
con.close()
print('Connection closed.')